In [1]:
import os
import json
import asyncio
import aiohttp
from pathlib import Path

In [ ]:
BASE = "https://sports.core.api.espn.com/v2/sports/basketball/leagues/mens-college-basketball"
SITE = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball"

SEASON = 2026
MAX_CONCURRENT_TEAMS = 10

OUTPUT_DIR = Path("./data/raw")
OUTPUT_DIR.mkdir(exist_ok=True)

group_cache = {}


async def fetch(session, url, params=None, retries=4, backoff=2):
    """
    Send one API request and return JSON.
    Retries transient ESPN/API errors like 504, 502, 503, 429.
    """
    for attempt in range(retries):
        try:
            async with session.get(url, params=params) as response:
                if response.status in [429, 500, 502, 503, 504]:
                    if attempt < retries - 1:
                        sleep_time = backoff ** attempt
                        print(
                            f"Retrying {response.status} for {url} "
                            f"attempt {attempt + 1}/{retries}, sleeping {sleep_time}s"
                        )
                        await asyncio.sleep(sleep_time)
                        continue

                response.raise_for_status()
                return await response.json(content_type=None)

        except Exception as error:
            if attempt < retries - 1:
                sleep_time = backoff ** attempt
                print(
                    f"Retrying error for {url} "
                    f"attempt {attempt + 1}/{retries}, sleeping {sleep_time}s: {error}"
                )
                await asyncio.sleep(sleep_time)
            else:
                raise

    return {}

async def fetch_or_empty(session, url, params=None, default=None):
    """
    Fetch JSON, but return a default object if the request fully fails.
    Useful for schedules so one timeout does not kill the whole team.
    """
    if default is None:
        default = {}

    try:
        return await fetch(session, url, params=params)
    except Exception as error:
        print(f"Failed after retries: {url} params={params} error={error}")
        return default
    
async def get_group(session, url):
    if not url:
        return {}

    if url not in group_cache:
        group_cache[url] = asyncio.ensure_future(fetch(session, url))

    return await group_cache[url]


async def fetch_all_team_refs(session):
    first = await fetch(
        session,
        f"{BASE}/seasons/{SEASON}/teams",
        params={"limit": 100}
    )

    all_refs = list(first.get("items", []))
    page_count = first.get("pageCount", 1)

    if page_count > 1:
        pages = await asyncio.gather(*[
            fetch(
                session,
                f"{BASE}/seasons/{SEASON}/teams",
                params={"limit": 100, "page": page}
            )
            for page in range(2, page_count + 1)
        ])

        for page in pages:
            all_refs.extend(page.get("items", []))

    return all_refs


async def process_team(session, item, semaphore):
    async with semaphore:
        try:
            team_ref = item["$ref"]

            team = await fetch(session, team_ref)
            team_id = str(team["id"])

            group_rows = []

            group_url = team.get("groups", {}).get("$ref", "")
            if group_url:
                group = await get_group(session, group_url)

                group_rows.append({
                    "group_ref": group_url,
                    "season": SEASON,
                    "raw_json": group
                })

                parent_url = group.get("parent", {}).get("$ref", "")
                if parent_url:
                    parent_group = await get_group(session, parent_url)

                    group_rows.append({
                        "group_ref": parent_url,
                        "season": SEASON,
                        "raw_json": parent_group
                    })

            record, roster, reg_schedule, post_schedule = await asyncio.gather(
                fetch_or_empty(
                    session,
                    team["record"]["$ref"],
                    default={"items": []}
                ),
                fetch_or_empty(
                    session,
                    f"{BASE}/seasons/{SEASON}/teams/{team_id}/athletes",
                    default={"items": []}
                ),
                fetch_or_empty(
                    session,
                    f"{SITE}/teams/{team_id}/schedule",
                    params={"season": SEASON, "seasontype": 2},
                    default={"events": []}
                ),
                fetch_or_empty(
                    session,
                    f"{SITE}/teams/{team_id}/schedule",
                    params={"season": SEASON, "seasontype": 3},
                    default={"events": []}
                ),
            )

            athlete_rows = []

            athlete_results = await asyncio.gather(*[
                fetch(session, athlete_ref["$ref"])
                for athlete_ref in roster.get("items", [])
            ], return_exceptions=True)

            for athlete in athlete_results:
                if isinstance(athlete, Exception):
                    continue

                athlete_rows.append({
                    "athlete_id": str(athlete.get("id")),
                    "team_id": team_id,
                    "season": SEASON,
                    "raw_json": athlete
                })

            return {
                "team": {
                    "team_id": team_id,
                    "season": SEASON,
                    "team_ref": team_ref,
                    "raw_json": team
                },
                "record": {
                    "team_id": team_id,
                    "season": SEASON,
                    "raw_json": record
                },
                "roster": {
                    "team_id": team_id,
                    "season": SEASON,
                    "raw_json": roster
                },
                "schedules": [
                    {
                        "team_id": team_id,
                        "season": SEASON,
                        "season_type": 2,
                        "raw_json": reg_schedule
                    },
                    {
                        "team_id": team_id,
                        "season": SEASON,
                        "season_type": 3,
                        "raw_json": post_schedule
                    }
                ],
                "athletes": athlete_rows,
                "groups": group_rows
            }

        except Exception as error:
            print(f"Error processing team {item.get('$ref')}: {error}")
            return None


def write_jsonl(file_path, rows):
    with open(file_path, "w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row) + "\n")


async def main():
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_TEAMS)
    connector = aiohttp.TCPConnector(limit=100)
    timeout = aiohttp.ClientTimeout(total=30)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        team_refs = await fetch_all_team_refs(session)
        print(f"Found {len(team_refs)} teams")

        results = await asyncio.gather(*[
            process_team(session, item, semaphore)
            for item in team_refs
        ], return_exceptions=True)

    team_rows = []
    record_rows = []
    roster_rows = []
    schedule_rows = []
    athlete_rows = []
    group_rows = []

    errors = 0

    for result in results:
        if result is None or isinstance(result, Exception):
            errors += 1
            continue

        team_rows.append(result["team"])
        record_rows.append(result["record"])
        roster_rows.append(result["roster"])
        schedule_rows.extend(result["schedules"])
        athlete_rows.extend(result["athletes"])
        group_rows.extend(result["groups"])

    # Deduplicate group rows by group_ref
    deduped_groups = {}
    for row in group_rows:
        deduped_groups[row["group_ref"]] = row

    group_rows = list(deduped_groups.values())

    write_jsonl(OUTPUT_DIR / "espn_team.jsonl", team_rows)
    write_jsonl(OUTPUT_DIR / "espn_record.jsonl", record_rows)
    write_jsonl(OUTPUT_DIR / "espn_roster.jsonl", roster_rows)
    write_jsonl(OUTPUT_DIR / "espn_schedule.jsonl", schedule_rows)
    write_jsonl(OUTPUT_DIR / "espn_athlete.jsonl", athlete_rows)
    write_jsonl(OUTPUT_DIR / "espn_group.jsonl", group_rows)

    print("Done writing JSONL files.")
    print(f"Teams: {len(team_rows)}")
    print(f"Records: {len(record_rows)}")
    print(f"Rosters: {len(roster_rows)}")
    print(f"Schedules: {len(schedule_rows)}")
    print(f"Athletes: {len(athlete_rows)}")
    print(f"Groups: {len(group_rows)}")
    print(f"Errors: {errors}")
    print(f"Files saved in: {OUTPUT_DIR.resolve()}")


# In a normal Python script:
# asyncio.run(main())

# In Jupyter:
await main()

Found 1105 teams
Done writing JSONL files.
Teams: 1105
Records: 1105
Rosters: 1105
Schedules: 2210
Athletes: 17266
Groups: 34
Errors: 0
Files saved in: C:\Users\Owner\Desktop\bus32120\bus32120-final-project\espn_json_output
